Read warptemplates for some class and look at properties of the sample:
- Do we end up having a sufficient number of (good) sn / template combos? (or is the set too narrow)?
- Look at the peak colors of all models and fit some distribution.
- Find coefficients to standardize etc. 

In [ ]:
import os, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import exponnorm
from pathlib import Path
from datetime import datetime
import pickle
import sncosmo

In [ ]:
from warptemplate import WarpfitTemplateLoader, add_warpclasses, register_all

In [ ]:
register_all()

### Inspect the base warp template
How many combos?

In [ ]:
# Which warp class to parse for?
# Category: (n)arrow, (e)xtended, (w)ide or (a)ll?
category = 'n'
# Id for class to run for
cid = 11
# Which version of data files
v = '4'

In [ ]:
# Vetoed combos (peak color totally off)
toskip = []
#toskip.append( 'ZTF21aanefkx_snana-2007lb+host' )      # Extreme color - must have gone wrong <

In [ ]:
n_classlist = [
    'SN IIP', 'SN Ia-91T', 'SN IIn', 'SN Ib/c', 'SN Ibn', 'SN Ia-pec', 'SLSN-I', 
    'SN Ic', 'SN Ic-BL', 'SN II', 'SLSN-II', 'SN Iax', 'SN Ia-91bg', 'SN Ia-CSM', 
    'SN Ia-SC', 'SN Ib', 'SN IIb',
]
e_classlist = [
    'SN Ib/c (e)', 'SLSN (e)'
]
w_classlist = [
    'SLSN (w)', 'SN II (w)', 'SN Ib/c (w)',
    'SN Ia (w)', 'SN Ia-pec (w)',
]
a_classlist = [
    'SN Ia (a)', 'SN CC (a)'
]

In [ ]:
if category=='n':
    class_name = n_classlist[cid]
print('Reading class', class_name)

# Load BTS information - mainly used to collect correct BTS name classes 
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')
df_bts = add_warpclasses(df_bts, purge=True)
classlist = list(set(df_bts['type_'+category]))
#class_name = n_classlist[cid]
#class_name = e_classlist[cid]
#class_name = w_classlist[cid]
class_name = a_classlist[cid]
#print('Target class {} from category {}'.format(class_name, 'type_'+category))
#classlist

In [ ]:
# Parameters for fit template retrieval
exclude_input = [] # Will reject any warptemplate containing any of these (either as sn or template basis)
# How to define templates?
# - How many templates per sn basis? 
#      * if 'all' it will return one copy of each template, 
#.     * if int it will return that many, drawn according to the template probability, 
#      * if -int it will return that many copies drawn from a uniform probabilitiy
#      Note: draws made with replacement, so multiple copies can be returned if int is larger than the available number of templates (often 3)
template_selection = 'all'    # Use the same number per SN to keep balance 
# - How many sn basis?
#.     * if 'all', take one of each
#.     * if an int, draw these randomly (with replacement)
#.     Note: how many templates are returned is decided by the above parameter.
snbasis_selection = 'all'
# Which fit quality to require?
min_fit_quality = None

warpdir = '/Users/jnordin/data/models/sncosmo/warpmod'

In [ ]:
warploader = WarpfitTemplateLoader(warpdir, version=v)

In [ ]:
tcounting = {}
for quality in ['gold', 'silver', 'bronze']:
    print('running quality', quality)
    templates = warploader.get_templates(
        fitclass=class_name,
        exclude_input = exclude_input, 
        template_selection=template_selection,
        snbasis_selection=snbasis_selection,
        min_fit_quality=quality,
        random_seed=42
    )
    # Analyse the results
    tmodels = [
        template['model'].description for template in templates
    ]
    tcounting[quality+'_nbr_models'] =  len(tmodels)
    tcounting[quality+'_nbr_sne'] =  len( set([ tmodel.split('_')[0] for tmodel in tmodels]) )
    tcounting[quality+'_nbr_templates']:  len( set([ tmodel.split('_')[1] for tmodel in tmodels]) )

In [ ]:
tcounting

In [ ]:
# Generate templates for studies

In [ ]:
templates = warploader.get_templates(
    fitclass=class_name,
    exclude_input = exclude_input, 
    template_selection=template_selection,
    snbasis_selection=snbasis_selection,
    min_fit_quality=min_fit_quality,
    random_seed=42
)

In [ ]:
templates[0]

In [ ]:
class_name

In [ ]:
# Which colors do we use for peak estimate
# do not remember how we chose this?
# for the observed data we of course only have ztf bands is it not easier to do with ztfg-ztfr?
#Try as far as possible ...
#colband = ['bessellb','bessellv']
# For some templates this does not seem to work?
#if class_name in [ 
#    'SN Ia-91T', 'SN Ia-pec', 'SLSN-I', 'SLSN-II', 'SN Ia-SC', 'SLSN (e)', 
#    'SLSN (w)', 'SN Ia (w)', 'SN Ia-pec (w)', 'SN Ia (a)', 'SN CC (a)']:
#    colband = ['ztfg','ztfr']
    
#if template_class_id in [3,9]:
#    # SLSN - templates start redder, using different bands
#    # but why also 91t?
#    colband = ['ztfr','ztfi']
#else:
#    colband = ['ztfg','ztfr']
#colband = ['bessellb','bessellv']


# When templates do not cover this, use bessel
#if class_name in [ 
#    'SN Iax'
#]:
#    colband = ['ztfr','ztfi']
#    colband = 


# Ok, as long as we evalute model colors at z 0, g-r seems to work.
colband = ['ztfg','ztfr']



In [ ]:
cols = {}
obscols = {}
peakphases = {}
for t in templates:
    # So - the current method in peakfitting_gp first fits the peak in each individual band
    # and then gets the color from the respective peaks. So do the same for the templates
    # We could also use the actual fitted color? Maybe compare both?
    #print(t['peak_gp_ztfg-ztfr'])
    #continue

    if t['model'].description in toskip:
        print('skipping ...', t['model'].description)
        continue

    # Lookup color from the warpmodels
    # What is the peak phase
    t0_0 = t['model'].source.peakphase(colband[0])
    t0_1 = t['model'].source.peakphase(colband[1])
    peakphases[t['model'].description] = {
        colband[0]:t0_0,
        colband[1]:t0_1,        
    }
    # Evaluate color at z 0?
    t['model'].set(z=0)

    cols[t['model'].description] = t['model'].bandmag(colband[0], 'ab', t0_0)-t['model'].bandmag(colband[1], 'ab', t0_1)

    # Also keep the measured values if existing
    obscols[t['model'].description] = t.get('peak_gp_ztfg-ztfr', None)
    

In [ ]:
_, bins, __ = plt.hist( cols.values(),bins=10, label='templates' )
#plt.xlim([-1,2])
plt.hist( obscols.values(),bins=bins, alpha=0.5, label='observed' )
plt.legend()

So it looks like these can be faaairly well behaved, described as some gaussian and then a tail to redder. So could fit one of these for each subclass? Then (i) fit a gaussian + exponential for each class (ii) record for each template the g-r color and the difference compared with the gaussian peak. This will be the initial Ebv diff. (iii) When simulating, draw a value from the combined guass + exp as the observed peak color (iv) set the template host color to this value and use for simulations.

#### Filtering
Differnt options for removing outliers prior to fitting the distribution

In [ ]:
def iqr_filter(data, k=1.5):
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return data[(data >= lower) & (data <= upper)]
def central_mask(data, lower_pct=5, upper_pct=95):
    lo, hi = np.percentile(data, [lower_pct, upper_pct])
    return data[(data >= lo) & (data <= hi)]

In [ ]:
q1, q3 = np.percentile(list(cols.values()), [25, 75])

In [ ]:
d1 = np.array(list(cols.values()))[np.array(list(cols.values()))>-np.inf]

In [ ]:
len(d1)

In [ ]:
# Below follows a recommendation for asymmetric rejection, probably cannot claim ..
# Step 1: remove extreme garbage only
d2 = iqr_filter(d1, k=3)

In [ ]:
len(d2)

In [ ]:
core = central_mask(d2, 5, 95)

In [ ]:
len(core)

In [ ]:
mu, sigma = np.mean(core), np.std(core)

In [ ]:
print(mu, sigma)

In [ ]:
# Step 3: asymmetric clipping
d3 = d2[(d2 > mu - 5*sigma) & (d2 < mu + 5*sigma)]

In [ ]:
len(d3)

In [ ]:
_, bins, __ = plt.hist(d1,bins=50)
_ = plt.hist(d2,bins=bins)
_ = plt.hist(d3,bins=bins)

In [ ]:
# What are we running with? depend on the sample size to some extent
if len(cols.values())<5:
    mydata = d1      # Only rempve Nan
else:
    #mydata = d2     # Mild filtering
    mydata = d3     # Five sigma rejection

# Do we need filtering? so far not ....
# So all of this for nothing - we kan skip the outlier rejection for the final color study?
mydata = d1
#if class_name in ['SLSN-II']:    # One large outlier
#    mydata = d2

In [ ]:

def fit_emg_and_store(data, model_name, col1, col2, outfile="warptemplate_v4_color_fits.csv", also_store={}):
    # Fit
    K, loc, scale = exponnorm.fit(data)

    # Create row
    result = {
        "model": model_name,
        "K": K,
        "loc": loc,
        "scale": scale,
        "color1": col1,
        "color2": col2,
        "n": len(data)
    }
    result.update( also_store )
    result["timestamp"] = datetime.utcnow().isoformat()

    # Convert to DataFrame
    df = pd.DataFrame([result])

    # Append to file
    file = Path(outfile)
    if file.exists():
        df.to_csv(file, mode="a", header=False, index=False)
    else:
        df.to_csv(file, index=False)

    return K, loc, scale

In [ ]:
K, loc, scale = fit_emg_and_store(mydata, class_name, colband[0], colband[1], also_store=tcounting )

In [ ]:
print(K, loc, scale)

In [ ]:
# Construct how many bins
if len(mydata)<30:
    bins = 5
elif len(mydata)<120:
    bins = 10
else:
    bins = 20

In [ ]:

# Plot ranges
s, e = min(mydata), max(mydata)

s = min(s, loc-0.8)
e = max(e, loc+1.2)


# -------------------
# Smooth PDF
# -------------------
x = np.linspace(s, e, 1000)
pdf = exponnorm.pdf(x, K, loc=loc, scale=scale)

# -------------------
# Style (publication-ready)
# -------------------
plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.2,
    "figure.dpi": 150,
})

fig, ax = plt.subplots(figsize=(6, 4))

# Histogram
ax.hist(mydata, bins=bins, density=True,
        alpha=0.5, color="steelblue",
        edgecolor="black", linewidth=0.5,
        label="Data")

# Fit curve
ax.plot(x, pdf, color="darkred", lw=2.5,
        label="EMG fit")

# Labels
ax.set_xlabel("Peak g-R (ZTF mag)")
ax.set_ylabel("Relative Frequency ")

# Legend
#ax.legend(frameon=False)

# Clean up spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

text = (
    f"K = {K:.2f}\n"
    f"$\\mu$ = {loc:.2f}\n"
    f"$\\sigma$ = {scale:.2f}"
)

ax.text(0.98, 0.95, text,
        transform=ax.transAxes,
        ha="right", va="top",
        bbox=dict(boxstyle="round", fc="white", ec="gray"))


# Tight layout
plt.tight_layout()

plt.savefig("/Users/jnordin/tmp/warpfigs/emg_fit_{}.png".format(class_name.replace("/","")), bbox_inches="tight",dpi=300)

plt.show()

### Individual color correction
As done in warpcoeff_distcolcorr

In [ ]:
# Inspect prediction - this we only did to visually check which of the two 
# emg parameters best corresponds to peak. The answer might be none of them, or that it varies
from scipy.stats import exponnorm
emg_dist = exponnorm(K, loc, scale)
dsim = emg_dist.rvs(size=1000000, random_state=41)
plt.figure(figsize=(5,3))
_ = plt.hist(dsim,bins=100)
plt.axvline(x=loc,color='red')
plt.axvline(x=loc+scale,color='green')

In [ ]:
# Now we wish to loop through all sncosmo templates and:
# - measure the template peak phase color
# - calculate which extinction would need to be applied to this template to achieve the class color 
# - create a modified version of the template warp coefficients that lead to the desired color
# - store the new version 

In [ ]:
    def color_with_ebv(warped_model, ebv, rv, band1, band2, t0_1, t0_2):
        warped_model.set(hostebv=ebv, hostr_v=rv)
        return (
            warped_model.bandmag(band1, 'ab', t0_1)
            - warped_model.bandmag(band2, 'ab', t0_2)
        )

In [ ]:
import sncosmo
from scipy.optimize import brentq
from scipy.optimize import minimize_scalar

In [ ]:
ebvout = {}
colfits = []
for k, t in enumerate(templates):

    # This 
    mod = t['model']
    modid = mod.description
    source = mod.source

    if modid in toskip:
        print('... skipping', modid)
        continue


    # Help functions
    warped_model = sncosmo.Model(
        source=source,
        effects=[sncosmo.CCM89Dust()],
        effect_names=['host'],
        effect_frames=['rest']
    )
    # How to do with readshift? Are we talking about observed peak color, or restframe peak color?
#    warped_model.set(z=mod.get('z'))
    warped_model.set(z=0)

    # We first determine the native peak col
#    natcol = color_with_ebv(warped_model, 0, 3.1, 
#                            colband[0], colband[1], 
#                            peakphases[modid][colband[0]], peakphases[modid][colband[1]],
#                           )
    natcol = cols[modid]
    print(k, modid, natcol)

    def fit_function( ebv ):
        return np.abs( color_with_ebv( 
                            warped_model, ebv, 3.1, 
                            colband[0], colband[1], 
                            peakphases[modid][colband[0]], peakphases[modid][colband[1]],
                        )-target_col
                     )
    
    # We now generate and loop through a number of simulated peak colors 
    for target_col in list(emg_dist.rvs(size=100)):
        # Determine which ebv is needed 
        # Look for the dust color ebv minimizing the difference
        try:
#            ebv_solution = brentq(fit_function, -5.0, 5.0, maxiter=100, )
            ebv_solution = minimize_scalar(fit_function)#, bounds=(-5.0, 5.0), method='bounded')

            if ebv_solution.success:

                res = color_with_ebv( 
                            warped_model, ebv_solution.x, 3.1, 
                            colband[0], colband[1], 
                            peakphases[modid][colband[0]], peakphases[modid][colband[1]],
                        )
#                print('got fit: target {:.2} from native col {:.2} fitted ebv {:.2} yield color {:.2} with diff {:.2f}'.format(
#                    target_col, natcol, ebv_solution.x, res, res-target_col )
#                     )

                
                # Store for analysis
                colfits.append(
                    {
                        'k': k,
                        'sn': modid,
                        'z': mod.get('z'),
                        'natcol': natcol,
                        'targetcol': target_col,
                        'ebvfit': ebv_solution.x
                    }
                )
            else:
                print('fit not success', target_col, natcol, natcol-target_col)
                    
        except KeyError:
            # fallback if root not bracketed
            print('fit fail', target_col, natcol, natcol-target_col)
            ebv_solution = -99
#    break

In [ ]:
dfcol = pd.DataFrame.from_dict( colfits )

In [ ]:
plt.scatter( dfcol['targetcol'], dfcol['ebvfit'] )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = dfcol['targetcol'] - dfcol['natcol']
y = dfcol['ebvfit']
z = dfcol['natcol']

# Fit quadratic
if class_name in ['SLSN-I','SLSN-II','SLSN (e)','SLSN (w)']:
    # Some peak lc fits fail
    mask = (dfcol['natcol']>-1)
    coeffs = np.polyfit(x[mask], y[mask], 3)
elif class_name in ['SN Ic-BL']:
    # Some peak lc fits fail
    mask = (dfcol['natcol']<1)
    coeffs = np.polyfit(x[mask], y[mask], 3)
else:
    coeffs = np.polyfit(x, y, 3)
    
poly = np.poly1d(coeffs)

# Generate smooth curve
x_fit = np.linspace(x.min(), x.max(), 300)
y_fit = poly(x_fit)

In [ ]:
class_name

In [ ]:
plt.figure(figsize=(7,5))

hb = plt.hexbin(x, y, gridsize=60, cmap='viridis', bins='log')
plt.colorbar(hb, label='log(N)')

plt.plot(x_fit, y_fit, color='red', linewidth=2, label='Color fit')

plt.grid(linestyle="--", alpha=0.3)
plt.legend()

plt.xlabel(r"$(g-R)_{draw}$ - $(g-R)_{template}$")
plt.ylabel("E(B-V)")
plt.tight_layout()
plt.savefig('distcolcorr_{}.pdf'.format(class_name.replace("/","")),dpi=300)
plt.show()

### Construct color information warpfiles

                "model_colors": dict    # Fit parameters of the original template fit (e.g. for color harmonization)
                    {
                        'K': 0.9079106174117336,
                        'loc': 0.1250977286061127,
                        'scale': 0.1514897618587996,
                        'color1': 'ztfg',
                        'color2': 'ztfr',
                        'ebv_corr_func': [-0.0001, 0.01, 0.5]  # polynomial coefficients to map from drawn color to E(B-V) correction
                    },



In [ ]:
model_colors = {
    'K': K, 
    'loc': loc, 
    'scale': scale, 
    'color1': colband[0],
    'color2': colband[1],
    'ebv_corr_func': list(coeffs)
}

In [ ]:
model_colors

In [ ]:
# Ok, so this information will be the same for each class - maybe store somwhere in code?
# What we would want in the warpfiles is the calculated peak color

In [ ]:
data = warploader._cache[class_name.replace("/","")]

In [ ]:
len(data)

In [ ]:
toskip

In [ ]:
newdata = {}
for snbase, snwarplist in data.items():
    newdata[snbase] = []
    for snwarp in snwarplist:
        mname = snwarp['id']+'_'+snwarp['model']
        if mname+'+host' in toskip:
            print('skip comb:', mname)
            continue
        snwarp['peakcol'] = cols[mname+'+host']
        print(mname, natcol)
        newdata[snbase].append( snwarp )

In [ ]:
warpdata = {
    'warpcoeff': newdata,
    'model_colors': model_colors,
}

In [ ]:
storefile = '/Users/jnordin/data/models/sncosmo/warpmod/warpcoeffs_v3_'+re.sub(r'/', '', class_name)+'.pkl'

In [ ]:
with open(storefile, 'wb') as file:
    pickle.dump(warpdata, file)